In [1]:
!pip install qonnx onnxruntime opencv-python numpy


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import cv2

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes

In [3]:
def preprocess_image(image_path, input_size=(416, 416)):
    """
    Preprocess image for YOLO inference - matches train.py preprocessing.
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not load image from {image_path}")
    
    original_shape = image.shape[:2]  # (height, width)
    
    # Resize
    resized = cv2.resize(image, input_size)
    
    # Convert BGR to RGB
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    
    # Normalize to [0,1] - matches train.py: imgs.float() / 255.0
    normalized = rgb.astype(np.float32) / 255.0
    
    # Transpose to CHW format
    chw = np.transpose(normalized, (2, 0, 1))
    
    # Add batch dimension
    batched = np.expand_dims(chw, axis=0)
    
    return batched, original_shape, image

In [4]:
def sigmoid(x):
    """Numerically stable sigmoid function."""
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

In [5]:
def xywh2xyxy(boxes):
    """Convert boxes from [x_center, y_center, w, h] to [x1, y1, x2, y2]."""
    xy = boxes[..., :2]
    wh = boxes[..., 2:4]
    xy1 = xy - wh / 2
    xy2 = xy + wh / 2
    return np.concatenate([xy1, xy2], axis=-1)

In [6]:
def box_iou(box1, box2):
    """
    Compute IoU between two sets of boxes.
    Boxes are in [x1, y1, x2, y2] format.
    """
    # Expand dimensions for broadcasting
    box1 = np.expand_dims(box1, axis=1)  # [N, 1, 4]
    box2 = np.expand_dims(box2, axis=0)  # [1, M, 4]
    
    # Intersection coordinates
    inter_x1 = np.maximum(box1[..., 0], box2[..., 0])
    inter_y1 = np.maximum(box1[..., 1], box2[..., 1])
    inter_x2 = np.minimum(box1[..., 2], box2[..., 2])
    inter_y2 = np.minimum(box1[..., 3], box2[..., 3])
    
    # Intersection area
    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # Union area
    box1_area = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    box2_area = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    union_area = box1_area + box2_area - inter_area
    
    # IoU
    iou = inter_area / (union_area + 1e-6)
    return iou

In [7]:
def non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45):
    """
    Performs Non-Maximum Suppression (NMS) on inference results.
    Based on utils.general.non_max_suppression from your code.
    
    Returns:
         list of detections, on (n,6) tensor per image [xyxy, conf, cls]
    """
    xc = prediction[..., 4] > conf_thres  # candidates
    
    # Settings
    max_wh = 7680  # maximum box width and height
    max_nms = 30000  # maximum number of boxes into torchvision.ops.nms()
    
    output = []
    for xi, x in enumerate(prediction):  # image index, image inference
        # Apply constraints
        x = x[xc[xi]]  # confidence
        
        # If none remain process next image
        if not x.shape[0]:
            output.append(np.zeros((0, 6)))
            continue
        
        # Compute conf
        x[:, 5:] *= x[:, 4:5]  # conf = obj_conf * cls_conf
        
        # Box (center x, center y, width, height) to (x1, y1, x2, y2)
        box = xywh2xyxy(x[:, :4])
        
        # Detections matrix nx6 (xyxy, conf, cls)
        conf = np.max(x[:, 5:], axis=1, keepdims=True)
        j = np.argmax(x[:, 5:], axis=1).reshape(-1, 1)
        x = np.concatenate((box, conf, j.astype(np.float32)), axis=1)[conf.reshape(-1) > conf_thres]
        
        # Check shape
        n = x.shape[0]  # number of boxes
        if not n:
            output.append(np.zeros((0, 6)))
            continue
        
        # Sort by confidence
        x = x[x[:, 4].argsort()[::-1][:max_nms]]
        
        # Batched NMS
        c = x[:, 5:6] * max_wh  # classes
        boxes, scores = x[:, :4] + c, x[:, 4]  # boxes (offset by class), scores
        
        # NMS
        keep = []
        indices = np.arange(x.shape[0])
        
        while boxes.shape[0]:
            # Pick detection with highest confidence (already sorted, so index 0)
            keep.append(indices[0])
            
            if boxes.shape[0] == 1:
                break
            
            # Compute IoU with remaining boxes
            ious = box_iou(boxes[0:1], boxes[1:])
            
            # Remove boxes with high IoU
            mask = ious[0] < iou_thres
            boxes = boxes[1:][mask]
            scores = scores[1:][mask]
            indices = indices[1:][mask]
        
        if keep:
            keep_indices = np.array(keep)
            output.append(x[keep_indices])
        else:
            output.append(np.zeros((0, 6)))
    
    return output

In [8]:
def scale_coords(img1_shape, coords, img0_shape):
    """
    Rescale coords (xyxy).
    """
    gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])  # gain = old / new
    pad = ((img1_shape[1] - img0_shape[1] * gain) / 2, 
           (img1_shape[0] - img0_shape[0] * gain) / 2)  # wh padding
    
    coords[:, [0, 2]] -= pad[0]  # x padding
    coords[:, [1, 3]] -= pad[1]  # y padding
    coords[:, :4] /= gain
    
    # Clip boxes
    coords[:, [0, 2]] = np.clip(coords[:, [0, 2]], 0, img0_shape[1])  # x
    coords[:, [1, 3]] = np.clip(coords[:, [1, 3]], 0, img0_shape[0])  # y
    
    return coords

In [9]:
def postprocess_yolo_output(outputs, anchors_config, strides, nc, input_shape):
    """
    Process raw YOLO outputs from FINN-exported model.
    
    Args:
        outputs: Dictionary with keys ['864', '868', '872'] containing raw outputs
        anchors_config: List of anchor boxes for each layer
        strides: List of strides for each layer
        nc: Number of classes
        input_shape: Input image shape (H, W)
    
    Returns:
        Predictions in format [batch, num_predictions, nc+5]
    """
    # Map outputs to detection layers based on grid size
    # Larger grid = smaller stride = P3, smaller grid = larger stride = P5
    output_items = []
    for key in outputs.keys():
        output = outputs[key]
        if output.ndim == 4:
            _, _, h, _ = output.shape
            output_items.append((h, key, output))
    
    # Sort by grid size: largest to smallest (P3 -> P4 -> P5)
    output_items.sort(reverse=True)
    
    z = []
    
    for idx, (_, key, output) in enumerate(output_items[:3]):
        # Get layer config
        anchors = np.array(anchors_config[idx], dtype=np.float32)  # [na, 2]
        stride = strides[idx]
        
        # Reshape output: [batch, na*no, ny, nx] -> [batch, na, no, ny, nx]
        bs, _, ny, nx = output.shape
        na = len(anchors)
        no = nc + 5
        
        # Reshape to [batch, na, no, ny, nx]
        output = output.reshape(bs, na, no, ny, nx)
        # Permute to [batch, na, ny, nx, no]
        output = output.transpose(0, 1, 3, 4, 2)
        
        # Create grid
        yv, xv = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
        grid = np.stack((xv, yv), axis=2).reshape(1, 1, ny, nx, 2).astype(np.float32)
        
        # Create anchor grid 
        anchor_grid = anchors.reshape(1, na, 1, 1, 2)
        
        # Apply sigmoid
        y = output.copy()
        
        # Standard sigmoid activation
        y_sigmoid = sigmoid(y)
        
        # xy offset - this gives absolute coordinates in the image
        xy = (y_sigmoid[..., 0:2] * 2.0 - 0.5 + grid) * stride
        # wh - this gives absolute width/height
        wh = (y_sigmoid[..., 2:4] * 2.0) ** 2 * anchor_grid
        
        # Apply sigmoid to objectness and class predictions
        obj_cls = y_sigmoid[..., 4:]
    
        # Concatenate - output format is [x_center, y_center, w, h, obj, cls]
        y_out = np.concatenate([xy, wh, obj_cls], axis=-1)
        
        # Reshape to [batch, num_anchors, no]
        z.append(y_out.reshape(bs, -1, no))
    
    # Concatenate all layers
    return np.concatenate(z, axis=1)

In [10]:
def draw_detections(image, detections, class_names=None):
    """Draw bounding boxes on image."""
    image_draw = image.copy()
    
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cls = int(cls)
        
        # Draw box
        color = (0, 255, 0)
        cv2.rectangle(image_draw, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        label = f"{class_names[cls] if class_names else cls}: {conf:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(image_draw, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
        cv2.putText(image_draw, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    
    return image_draw

In [11]:
def main():
    # Configuration from your YAML
    model_path = 'best.qonnx.onnx'
    image_path = 'test.jpg'
    input_size = (416, 416)
    
    # Model config
    nc = 1  # number of classes

    anchors_config = [
        [[10, 13], [16, 30], [33, 23]],      # P3/8
        [[30, 61], [62, 45], [59, 119]],     # P4/16
        [[116, 90], [156, 198], [373, 326]]  # P5/32
    ]
    strides = [8, 16, 32]
    
    # Inference parameters
    conf_thres = 0.25  # Lower threshold to see more detections
    iou_thres = 0.45
    
    # Class names
    class_names = ['Person']
    
    print(f"Model: {model_path}")
    print(f"Image: {image_path}")
    print(f"Input size: {input_size}")
    print(f"Confidence threshold: {conf_thres}")
    print(f"IoU threshold: {iou_thres}")
    
    # Load QONNX model
    print("\nLoading QONNX model...")
    model = ModelWrapper(model_path)
    
    # Infer shapes
    print("Inferring tensor shapes...")
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    
    # Get input info
    input_name = model.graph.input[0].name
    print(f"Input name: {input_name}")
    input_shape = model.get_tensor_shape(input_name)
    print(f"Expected input shape: {input_shape}")
    
    # Preprocess image
    print("Preprocessing image...")
    input_tensor, original_shape, original_image = preprocess_image(
        image_path, input_size
    )
    print(f"Input tensor shape: {input_tensor.shape}")
    print(f"Input tensor range: [{input_tensor.min():.3f}, {input_tensor.max():.3f}]")
    print(f"Original image shape: {original_shape}")
    
    # Create input dict
    input_dict = {input_name: input_tensor}
    
    # Execute inference
    print("Running inference...")
    output_dict = execute_onnx(model, input_dict)
    
    # Postprocess - convert raw outputs to predictions
    print("Postprocessing detections...")
    predictions = postprocess_yolo_output(
        output_dict, anchors_config, strides, nc, input_size
    )
    print(f"Predictions shape: {predictions.shape}")
    print(f"Number of anchor predictions: {predictions.shape[1]}")
    
    # Apply NMS
    print("Applying NMS...")
    print(f"Number of predictions > {conf_thres}: {(predictions[..., 4] > conf_thres).sum()}")
    
    detections = non_max_suppression(predictions, conf_thres, iou_thres)
    detections = detections[0] 
    
    print(f"\nFound {len(detections)} detections")
    
    if len(detections) > 0:
        # Scale boxes back to original image size
        detections[:, :4] = scale_coords(input_size, detections[:, :4], original_shape)
        
        # Print detections
        for i, det in enumerate(detections):
            x1, y1, x2, y2, conf, cls = det
            class_name = class_names[int(cls)] if class_names else int(cls)
            print(f"  Detection {i+1}: {class_name} ({conf:.3f}) at [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
        
        # Draw and save result
        result_image = draw_detections(original_image, detections, class_names)
        output_path = 'output_detections.jpg'
        cv2.imwrite(output_path, result_image)
        print(f"\nResult saved to {output_path}")
    else:
        print("\nNo detections found. Try:")

In [12]:
if __name__ == "__main__":
    main()

Model: best.qonnx.onnx
Image: test.jpg
Input size: (416, 416)
Confidence threshold: 0.25
IoU threshold: 0.45

Loading QONNX model...
Inferring tensor shapes...
Input name: quant_input.1
Expected input shape: [1, 3, 416, 416]
Preprocessing image...
Input tensor shape: (1, 3, 416, 416)
Input tensor range: [0.000, 0.898]
Original image shape: (416, 416)
Running inference...
Postprocessing detections...
Predictions shape: (1, 10647, 6)
Number of anchor predictions: 10647
Applying NMS...
Number of predictions > 0.25: 105

Found 5 detections
  Detection 1: Person (0.466) at [198.8, 204.8, 265.1, 389.9]
  Detection 2: Person (0.462) at [132.7, 81.7, 173.0, 241.0]
  Detection 3: Person (0.456) at [14.6, 146.2, 62.3, 320.5]
  Detection 4: Person (0.421) at [1.0, 304.1, 60.3, 414.6]
  Detection 5: Person (0.386) at [62.6, 151.7, 112.6, 309.4]

Result saved to output_detections.jpg
